# Assignment 1 - Personal chat with an Actor, a Critic and a revision loop

**Course:** AI Systems - Dr. Paul Avrahami | **Material:** Lecture 2, slides 19-21

This notebook builds the chat step by step:

| Part | What | Status |
|---|---|---|
| **A** | Background sources + Actor (answer generator) + Hebrew Gradio chat | next |
| B | Critic (evaluator) with a Pydantic `Evaluation` structured output | next |
| C | Generate -> evaluate -> revise loop (max 3 attempts) + run log | next |
| D | 5 scenarios | next |

**Provider:** Anthropic Claude, called through the **OpenAI Python SDK** (as in the labs) pointed at Anthropic's
OpenAI-compatible endpoint, with the Anthropic API key. No framework - direct orchestration in code.

### Setup (once)
1. In the repo root: `uv sync` (creates `.venv` with the dependencies from `pyproject.toml`).
2. Paste your key into `.env` in the repo root: `ANTHROPIC_API_KEY=sk-ant-...` (see `.env.example`).
3. In Cursor / VS Code select the kernel **`.venv` (Python 3.12)** and run the cells top to bottom.

## 0. Imports and environment

In [1]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import gradio as gr
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from pydantic import BaseModel  # used for the Critic's Evaluation model in Part B

c:\Users\User\Documents\cursor\agents_HW\Agents_HW_1_personal_chat\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load_dotenv() searches the current folder and its parents, so the .env in the repo root is found.
load_dotenv(override=True)

anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
if not anthropic_api_key:
    raise RuntimeError("ANTHROPIC_API_KEY is missing - add it to the .env file in the repo root (see .env.example).")
print(f"Anthropic API key exists and begins {anthropic_api_key[:10]}")  # never print the full key

# Same pattern as the labs: the OpenAI client, with base_url pointing at Claude's OpenAI-compatible endpoint
ANTHROPIC_BASE_URL = "https://api.anthropic.com/v1/"
openai = OpenAI(base_url=ANTHROPIC_BASE_URL, api_key=anthropic_api_key)

Anthropic API key exists and begins sk-ant-usr


## 1. Configuration

Model names are documented here, as the assignment requires. The same model can serve the Actor and the Critic,
with separate calls and separate prompts; change `CRITIC_MODEL` in Part B if you want a different one.

Calls use the lab pattern: `openai.chat.completions.create(model=..., messages=[{"role": "system", ...}, ...])`.

Notes on Anthropic's OpenAI-compatible endpoint:
* It is described as meant for testing and comparing models, not long-term production use.
* System messages are moved into Claude's `system` field. Keep one system message at the start of `messages`.
* Claude Opus 5.5 thinks by default, but the thinking isn't returned. `reasoning_effort` is ignored.
* `response_format` (structured output, Part B) accepts only `json_schema` with `strict: true`. Anything else returns a 400 error.
* No prompt caching.

In [ ]:
ACTOR_MODEL = "claude-opus-5-5"
CRITIC_MODEL = "claude-opus-5-5"   # Part B
MAX_TOKENS = 16000
MAX_ATTEMPTS = 3                   # Part C: up to 3 generation attempts per question, including the first one

PROJECT_DIR = Path.cwd()           # run the notebook from the personal_chat folder
DATA_DIR = PROJECT_DIR / "data"
LOG_DIR = PROJECT_DIR / "logs"
LOG_DIR.mkdir(exist_ok=True)

PERSON_NAME = "נועה לוי"            # the (sample) person the chat represents